In [0]:
landing_path = "/Volumes/fraud_mlops/bronze/paysim_landing"

display(dbutils.fs.ls(landing_path))

In [0]:
sample_df = spark.read.option('header',True).option('inferSchema',True).csv('/Volumes/fraud_mlops/bronze/paysim_landing/paysim_dataset')

sample_df.printSchema()

In [0]:
from pyspark.sql.types import StructType,StructField,IntegerType,StringType,DoubleType

paysim_schema = StructType([
    StructField('step',IntegerType(),True),
    StructField('type',StringType(),True),
    StructField('amount',DoubleType(),True),
    StructField('nameOrig',StringType(),True),
    StructField('oldbalanceOrg',DoubleType(),True),
    StructField('newbalanceOrig',DoubleType(),True),
    StructField('nameDest',StringType(),True),
    StructField('oldbalanceDest',DoubleType(),True),
    StructField('newbalanceDest',DoubleType(),True),
    StructField('isFraud',IntegerType(),True),
    StructField('isFlaggedFraud',IntegerType(),True)

])

In [0]:
raw_df = spark.read.format('csv').option('header',True).schema(paysim_schema).load('/Volumes/fraud_mlops/bronze/paysim_landing/paysim_dataset')

display(raw_df.limit(10))

In [0]:
from pyspark.sql.functions import current_timestamp,col

bronze_df = raw_df.withColumn('ingested_at',current_timestamp())\
    .withColumn('source_file', col('_metadata.file_path'))

display(bronze_df.limit(10))


In [0]:
bronze_df.write.mode('overwrite').format('delta').option('overwriteSchema',True).saveAsTable('fraud_mlops.bronze.paysim_transactions')



In [0]:
bronze_df.groupBy('type').count().orderBy('count', ascending=False).display()